In [63]:
import pandas as pd

df = pd.read_excel("dados_exportados_jira_adaptado.xlsx")

mask_resumo = df["Chave"].astype(str).str.strip().eq("Resumo")
df["MetaKey"] = df["Resumo"].where(mask_resumo, df["META_ID"])

df[(df['Chave'] != 'Resumo')]


,META_ID,Chave,Resumo,Nº_Meta,Meta_apuração,% de Cumprimento,Afeta as versões,Ano da Meta,Apurado no período,Cargo do Gerente Responsável,...,Subtarefas,Telefone do Gerente Responsável,Tipo de Meta,Unidade Gestora,Unidade de Medida,Valor Apurado,Valor da Meta,Versões de correção,Votos,MetaKey
0,ASPLAGMETA-2871,ASPLAGMETA-2884,"Apuração: TJMG 154 – Definir e preparar, até d...","TJMG 154 – Definir e preparar, até dezembro de...",[ASPLAGMETA-2884] Apuração: TJMG 154 – Definir...,NaN,Nenhum,2025.0,Apurado no período,NaN,...,NaN,NaN,NaN,NaN,Percentual,100,100.0,Nenhum,0,ASPLAGMETA-2871
1,ASPLAGMETA-2871,ASPLAGMETA-2883,Dezembro,"TJMG 154 – Definir e preparar, até dezembro de...",[ASPLAGMETA-2883] Dezembro,NaN,Nenhum,2025.0,NaN,NaN,...,NaN,NaN,NaN,NaN,Percentual,NaN,NaN,Nenhum,0,ASPLAGMETA-2871
2,ASPLAGMETA-2871,ASPLAGMETA-2882,Novembro,"TJMG 154 – Definir e preparar, até dezembro de...",[ASPLAGMETA-2882] Novembro,NaN,Nenhum,2025.0,NaN,NaN,...,NaN,NaN,NaN,NaN,Percentual,NaN,NaN,Nenhum,0,ASPLAGMETA-2871
3,ASPLAGMETA-2871,ASPLAGMETA-2881,Outubro,"TJMG 154 – Definir e preparar, até dezembro de...",[ASPLAGMETA-2881] Outubro,NaN,Nenhum,2025.0,NaN,NaN,...,NaN,NaN,NaN,NaN,Percentual,NaN,NaN,Nenhum,0,ASPLAGMETA-2871
4,ASPLAGMETA-2871,ASPLAGMETA-2880,Setembro,"TJMG 154 – Definir e preparar, até dezembro de...",[ASPLAGMETA-2880] Setembro,NaN,Nenhum,2025.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nenhum,0,ASPLAGMETA-2871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,ASPLAGMETA-1088,ASPLAGMETA-1883,Janeiro,TJMG 1g - Beneficiar 20.000 (vinte mil) pessoa...,[ASPLAGMETA-1883] Janeiro,NaN,Nenhum,2024.0,NaN,NaN,...,NaN,NaN,NaN,NaN,Pessoa,NaN,NaN,Nenhum,0,ASPLAGMETA-1088
996,ASPLAGMETA-1869,ASPLAGMETA-1882,"Apuração: TJMG 113 - Exercer, em 2024, o juízo...","TJMG 113 - Exercer, em 2024, o juízo de admiss...",[ASPLAGMETA-1882] Apuração: TJMG 113 - Exercer...,0,Nenhum,NaN,Apurado no período,NaN,...,NaN,NaN,NaN,NaN,Percentual,0,100.0,Nenhum,0,ASPLAGMETA-1869
997,ASPLAGMETA-1869,ASPLAGMETA-1881,Dezembro,"TJMG 113 - Exercer, em 2024, o juízo de admiss...",[ASPLAGMETA-1881] Dezembro,NaN,Nenhum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Percentual,0,100.0,Nenhum,0,ASPLAGMETA-1869
998,ASPLAGMETA-1869,ASPLAGMETA-1880,Novembro,"TJMG 113 - Exercer, em 2024, o juízo de admiss...",[ASPLAGMETA-1880] Novembro,0,Nenhum,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Percentual,0,100.0,Nenhum,0,ASPLAGMETA-1869


In [64]:
# propagar Macrodesafio dos resumos para as linhas filhas
macro_map = (
    df.loc[mask_resumo, ["MetaKey", "Macrodesafio"]]
      .dropna(subset=["Macrodesafio"])
      .assign(Macrodesafio=lambda x: x["Macrodesafio"].astype(str).str.strip())
      .replace({"Macrodesafio": {"": pd.NA}})
      .dropna(subset=["Macrodesafio"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Macrodesafio"]
 )
mask_macro_vazio = df["Macrodesafio"].isna() | df["Macrodesafio"].astype(str).str.strip().eq("")
df.loc[mask_macro_vazio, "Macrodesafio"] = df.loc[mask_macro_vazio, "MetaKey"].map(macro_map)

df['Macrodesafio']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999    NaN
Name: Macrodesafio, Length: 1000, dtype: object

In [65]:
# propagar Ano da Meta dos resumos para as linhas filhas
ano_map = (
    df.loc[mask_resumo, ["MetaKey", "Ano da Meta"]]
      .dropna(subset=["Ano da Meta"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Ano da Meta"]
 )
mask_ano_vazio = df["Ano da Meta"].isna()
df.loc[mask_ano_vazio, "Ano da Meta"] = df.loc[mask_ano_vazio, "MetaKey"].map(ano_map)

df['Ano da Meta']

0      2025.0
1      2025.0
2      2025.0
3      2025.0
4      2025.0
        ...  
995    2024.0
996       NaN
997       NaN
998       NaN
999       NaN
Name: Ano da Meta, Length: 1000, dtype: float64

In [66]:
# mover a coluna "MetaKey" para a primeira coluna
if 'MetaKey' in df.columns:
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('MetaKey')))
    df = df[cols]
else:
    raise KeyError("Coluna 'MetaKey' não encontrada em df")

In [67]:
# criar coluna "mes_apuracao" no formato "set-25" a partir de "Data de Apuração"
meses = {1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr', 5: 'mai', 6: 'jun',
         7: 'jul', 8: 'ago', 9: 'set', 10: 'out', 11: 'nov', 12: 'dez'}

dt = pd.to_datetime(df['Data de Apuração'], dayfirst=True, errors='coerce')

def formato_mes(dt_val):
    if pd.isna(dt_val):
        return pd.NA
    return f"{meses[dt_val.month]}-{str(dt_val.year)[-2:]}"

df['mes_apuracao'] = dt.apply(formato_mes)

# conferir resultado
df[['Data de Apuração', 'mes_apuracao']].head(10)

C:\Users\P0165559\AppData\Local\Temp\ipykernel_29328\1956484759.py:5: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(df['Data de Apuração'], dayfirst=True, errors='coerce')


,Data de Apuração,mes_apuracao
0,2026-01-20,jan-26
1,NaN,<NA>
2,NaN,<NA>
3,NaN,<NA>
4,NaN,<NA>
5,NaN,<NA>
6,NaN,<NA>
7,NaN,<NA>
8,NaN,<NA>
9,NaN,<NA>


In [68]:
import re

def extract_indicator(val):
    if pd.isna(val):
        return pd.NA
    s = str(val)
    pattern = r"\d{4}\s*\n\s*-\s*"
    if re.search(pattern, s):
        cleaned = re.sub(pattern, "", s, count=1).strip()
        return cleaned if cleaned else pd.NA
    return pd.NA

df['Indicador'] = df['Indicador Estratégico'].apply(extract_indicator)

# conferir algumas linhas
df[['Indicador Estratégico', 'Indicador']]

,Indicador Estratégico,Indicador
0,NaN,<NA>
1,NaN,<NA>
2,NaN,<NA>
3,NaN,<NA>
4,NaN,<NA>
...,...,...
995,NaN,<NA>
996,NaN,<NA>
997,NaN,<NA>
998,NaN,<NA>


In [69]:
# propagar Indicador dos resumos para as linhas filhas
indicador_map = (
    df.loc[mask_resumo, ["MetaKey", "Indicador"]]
      .dropna(subset=["Indicador"])
      .assign(Indicador=lambda x: x["Indicador"].astype(str).str.strip())
      .replace({"Indicador": {"": pd.NA}})
      .dropna(subset=["Indicador"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Indicador"]
 )
mask_indicador_vazio = df["Indicador"].isna() | df["Indicador"].astype(str).str.strip().eq("")
df.loc[mask_indicador_vazio, "Indicador"] = df.loc[mask_indicador_vazio, "MetaKey"].map(indicador_map)

df['Indicador']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999    NaN
Name: Indicador, Length: 1000, dtype: object

In [70]:
# extrair até dois códigos (1-2 dígitos) da coluna 'Iniciativas Estratégicas 2025', incluindo ocorrências após vírgula
def extract_iniciativa(val):
    if pd.isna(val):
        return pd.NA
    s = str(val)
    primeiro = re.search(r"\b(\d{1,2})\b", s)
    if not primeiro:
        return pd.NA
    codigos = [f"IE {primeiro.group(1)}"]
    seguintes = re.findall(r",\s*(\d{2})\b", s[primeiro.end():])
    if seguintes:
        codigos.extend(f"IE {codigo}" for codigo in seguintes)
    codigos_unicos = []
    for codigo in codigos:
        if codigo not in codigos_unicos:
            codigos_unicos.append(codigo)
    return ", ".join(codigos_unicos) if codigos_unicos else pd.NA

df['Iniciativa'] = df['Iniciativas Estratégicas 2025'].apply(extract_iniciativa)

# conferir resultad

In [71]:
# propagar Iniciativa dos resumos para as linhas filhas
iniciativa_map = (
    df.loc[mask_resumo, ["MetaKey", "Iniciativa"]]
      .dropna(subset=["Iniciativa"])
      .assign(Iniciativa=lambda x: x["Iniciativa"].astype(str).str.strip())
      .replace({"Iniciativa": {"": pd.NA}})
      .dropna(subset=["Iniciativa"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Iniciativa"]
 )
mask_iniciativa_vazia = df["Iniciativa"].isna() | df["Iniciativa"].astype(str).str.strip().eq("")
df.loc[mask_iniciativa_vazia, "Iniciativa"] = df.loc[mask_iniciativa_vazia, "MetaKey"].map(iniciativa_map)

df['Iniciativa']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999    NaN
Name: Iniciativa, Length: 1000, dtype: object

In [72]:
# propagar Unidade Gestora dos resumos para as linhas filhas
unidade_map = (
    df.loc[mask_resumo, ["MetaKey", "Unidade Gestora"]]
      .dropna(subset=["Unidade Gestora"])
      .assign(**{"Unidade Gestora": lambda x: x["Unidade Gestora"].astype(str).str.strip()})
      .replace({"Unidade Gestora": {"": pd.NA}})
      .dropna(subset=["Unidade Gestora"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Unidade Gestora"]
 )
mask_unidade_vazia = df["Unidade Gestora"].isna() | df["Unidade Gestora"].astype(str).str.strip().eq("")
df.loc[mask_unidade_vazia, "Unidade Gestora"] = df.loc[mask_unidade_vazia, "MetaKey"].map(unidade_map)

df['Unidade Gestora']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999    NaN
Name: Unidade Gestora, Length: 1000, dtype: object

In [73]:
# propagar Polaridade dos resumos para as linhas filhas
polaridade_map = (
    df.loc[mask_resumo, ["MetaKey", "Polaridade"]]
      .dropna(subset=["Polaridade"])
      .assign(Polaridade=lambda x: x["Polaridade"].astype(str).str.strip())
      .replace({"Polaridade": {"": pd.NA}})
      .dropna(subset=["Polaridade"])
      .drop_duplicates(subset=["MetaKey"], keep="first")
      .set_index("MetaKey")["Polaridade"]
 )
mask_polaridade_vazia = df["Polaridade"].isna() | df["Polaridade"].astype(str).str.strip().eq("")
df.loc[mask_polaridade_vazia, "Polaridade"] = df.loc[mask_polaridade_vazia, "MetaKey"].map(polaridade_map)

df['Polaridade']

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    NaN
997    NaN
998    NaN
999    NaN
Name: Polaridade, Length: 1000, dtype: object

In [74]:
# análise da coluna "Informação complementar" sem tags HTML
import re
from bs4 import BeautifulSoup

def remove_html(val):
    if pd.isna(val):
        return pd.NA
    soup = BeautifulSoup(str(val), 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA

df['Informação complementar texto'] = df['Informação complementar'].apply(remove_html)
col_info = df['Informação complementar texto']

summary_info = pd.Series({
    'total_registros': len(col_info),
    'valores_nao_nulos': int(col_info.notna().sum()),
    'valores_unicos_nao_nulos': int(col_info.dropna().nunique()),
    'strings_vazias': int((col_info.fillna('').astype(str).str.strip() == '').sum())
}, name='Resumo Informação complementar')


col_info_clean = col_info.dropna().astype(str).str.strip()
length_stats = col_info_clean.str.len().describe()


top_valores = col_info_clean.value_counts().head(10)


In [75]:
df.to_excel('arquivo_extração_jira.xlsx', index=False)

In [76]:
# Lista de colunas que você quer manter
colunas = [
    'Macrodesafio', 'MetaKey', 'Ano da Meta', 'Resumo', 'Indicador',
    'Unidade Gestora', 'Polaridade', 'Valor da Meta', 'Valor Apurado', 'Iniciativa',
    'Informação complementar texto'
]

# Definição do filtro (Ano 2025 E (Resumo Dezembro OU Apurado no período))
filtro = (df['Ano da Meta'] == 2025) & (df['Resumo'].isin(['Dezembro', 'Apurado no período']))

# Aplicação do filtro e seleção das colunas
df_teste = df.loc[filtro, colunas]


In [77]:

# unificar linhas por MetaKey: usar a linha "Apurado no período" como base
# e trazer "Informação complementar texto" da linha "Dezembro" quando existir
def unificar_por_metakey(df):
    registros = []
    for meta, grupo in df.groupby('MetaKey', sort=False):
        # base = linha "Apurado no período" se existir, senão primeira linha do grupo
        apurado = grupo[grupo['Resumo'] == 'Apurado no período']
        if not apurado.empty:
            base = apurado.iloc[0].to_dict()
            base_resumo = 'Apurado no período'
        else:
            base = grupo.iloc[0].to_dict()
            base_resumo = base.get('Resumo')

        # obter "Informação complementar texto" da linha "Dezembro" (primeiro não-nulo)
        dez = grupo[grupo['Resumo'] == 'Dezembro']
        info_compl = None
        if not dez.empty:
            s = dez['Informação complementar texto'].dropna()
            if not s.empty:
                info_compl = s.iloc[0]

        # fallback para o valor da base se não houver texto em Dezembro
        if info_compl is None:
            info_compl = base.get('Informação complementar texto')

        base['Informação complementar texto'] = info_compl
        base['Resumo'] = base_resumo  # garante que o resumo seja "Apurado no período" quando possível
        registros.append(base)

    return pd.DataFrame(registros)

df_teste_unificado = unificar_por_metakey(df_teste)
df_teste_unificado.reset_index(drop=True, inplace=True)


In [78]:
# ordenar considerando o número inicial de Macrodesafio, Indicador e MetaKey
def _leading_number(series):
    extracted = series.astype(str).str.extract(r"^(\d+)")[0]
    return pd.to_numeric(extracted, errors="coerce")

df_teste_unificado = (
    df_teste_unificado.assign(
        _macro_ord=_leading_number(df_teste_unificado["Macrodesafio"]),
        _indicador_ord=_leading_number(df_teste_unificado["Indicador"]),
        _metakey_ord=_leading_number(df_teste_unificado["MetaKey"]),
    )
    .sort_values(
        by=[
            "_macro_ord",
            "Macrodesafio",
            "_indicador_ord",
            "Indicador",
            "_metakey_ord",
            "MetaKey",
        ],
        kind="mergesort",
        na_position="last",
    )
    .drop(columns=["_macro_ord", "_indicador_ord", "_metakey_ord"])
    .reset_index(drop=True)
 )

In [79]:
df_teste_unificado.to_excel('teste_integração.xlsx', index=False)